In [2]:
include("../RayTracing.jl")

Main.RayTracing

In [3]:
function parse_vertex(line::AbstractString)::RayTracing.Pnt3
	parts = split(strip(line))[2:end]
	if length(parts) != 3
		throw(ArgumentError("Invalid vertex format: $line"))
	end
	return RayTracing.Pnt3(parse(Float64, parts[1]), parse(Float64, parts[2]), parse(Float64, parts[3]))
end

function parse_normal(line::AbstractString)::RayTracing.Nml3
	parts = split(strip(line))[2:end]
	if length(parts) != 3
		throw(ArgumentError("Invalid vertex format: $line"))
	end
	return RayTracing.Nml3(parse(Float64, parts[1]), parse(Float64, parts[2]), parse(Float64, parts[3]))
end

function parse_uv(line::AbstractString)::RayTracing.Pnt2
	parts = split(strip(line))[2:end]
	if length(parts) != 2
		throw(ArgumentError("Invalid vertex format: $line"))
	end
	return RayTracing.Pnt2(parse(Float64, parts[1]), parse(Float64, parts[2]))
end

function parse_face!(
    line::AbstractString,
    vertex_indices::Vector{Int},
    uv_indices::Vector{Int},
    normal_indices::Vector{Int}
)
	parts = split(strip(line))[2:end]
    
    if length(parts) != 3
        throw(ArgumentError("Invalid face format on line: $line"))
    end
	
	for part in parts
		indices = split(part, "/")
		push!(vertex_indices, parse(Int, indices[1]))
		
        if length(indices) > 1 && indices[2] != ""
            push!(uv_indices, parse(Int, indices[2]))
        end
		
		if length(indices) > 2 && indices[3] != ""
            push!(normal_indices, parse(Int, indices[3]))
        end
    end
end

function parse_obj2(
    file_path::AbstractString,
    object_to_world::RayTracing.Transformation, 
    reverse_orientation::Bool, 
    transform_swaps_handedness::Bool,
    alpha_mask::RayTracing.Maybe{RayTracing.Texture},
)
	vertices = RayTracing.Pnt3[]
    vertex_indices = Int[]

	uvs = RayTracing.Pnt2[]
    uv_indices = Int[]

	normals = RayTracing.Nml3[]
	normal_indices = Int[]
	
	open(file_path) do file
		for line in eachline(file)
			line = strip(line)
			if isempty(line) || startswith(line, "#")
				continue
			end
			
			parts = split(line)
			cmd = parts[1]
			
            if cmd == "v"
                push!(vertices, parse_vertex(line))
            elseif cmd == "vt"
                push!(uvs, parse_uv(line))
            elseif cmd == "vn"
                push!(normals, parse_normal(line))
            elseif cmd == "f"
                parse_face!(line, vertex_indices, uv_indices, normal_indices)
            elseif cmd == "mtllib"
                @warn "Skipping material: $line"
                continue
            elseif cmd == "usemtl"
                @warn "Skipping material: $line"
                continue
            else
                @assert false
            end
		end

        @assert mod(length(vertex_indices), 3) == 0
        @assert mod(length(uv_indices), 3) == 0
        @assert mod(length(normal_indices), 3) == 0
	end
	return (
        RayTracing.ShapeCore(object_to_world, RayTracing.Inv(object_to_world), reverse_orientation, transform_swaps_handedness), 
        length(vertex_indices)÷3,  #n_triangles
        vertices,
        vertex_indices,
        normals,
        normal_indices,
        uvs,
        uv_indices,
        alpha_mask
    )
end


parse_obj2 (generic function with 1 method)

In [4]:
(sc, 
n_triangles, 
vertices,
vertex_indices,
normals,
normal_indices,
uvs,
uv_indices,
alpha_mask) = parse_obj2(
    RayTracing.jmfp("/home/jmyslinski/random_stuff/PBRJ/test/plane.obj"),
    RayTracing.Translate(RayTracing.Pnt3(0,0,0)),
    false,
    false,
    nothing
)

┌ Warning: Skipping material: mtllib plane.mtl
└ @ Main /home/jmyslinski/random_stuff/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W1sdnNjb2RlLXJlbW90ZQ==.jl:86
┌ Warning: Skipping material: usemtl Material
└ @ Main /home/jmyslinski/random_stuff/PBRJ/src/notebooks/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W1sdnNjb2RlLXJlbW90ZQ==.jl:89


(Main.RayTracing.ShapeCore(Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), Main.RayTracing.Transformation([1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 1.0]), false, false), 2, Main.RayTracing.Pnt3[[-1.0, 0.0, -1.0], [1.0, 0.0, -1.0], [1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]], [1, 2, 3, 1, 2, 4], Main.RayTracing.Nml3[[0.0, 1.0, 0.0]], [1, 1, 1, 1, 1, 1], Main.RayTracing.Pnt2[[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]], [1, 2, 3, 1, 2, 4], nothing)

In [ ]:
# # plane.obj
# mtllib plane.mtl
# v -1.0 0.0 -1.0
# v 1.0 0.0 -1.0
# v 1.0 0.0 1.0
# v -1.0 0.0 1.0

# vt 0.0 0.0
# vt 1.0 0.0
# vt 1.0 1.0
# vt 0.0 1.0

# vn 0.0 1.0 0.0

# usemtl Material
# f 1/1/1 2/2/1 3/3/1 
# f 1/1/1 2/2/1 4/4/1

######
# SHOULD BECOME
######

# vertices = Pnt3[Pnt3(), Pnt3(), Pnt3(), Pnt3()]
# uvs = Pnt2[Pnt2(), Pnt2(), Pnt2(), Pnt2()]
# normals = Nml3[Nml3()]

# vertex_indices = Int64[1, 2, 3, 1, 2, 4]
# uv_indices = Int64[1, 2, 3, 1, 2, 4]
# normal_indices = Int64[1, 1, 1, 1, 1, 1]

# @assert mod(length(vertex_indices), 3) == 0
# @assert mod(length(uv_indices), 3) == 0
# @assert mod(length(normal_indices), 3) == 0

# n_triangles = length(vertex_indices) ÷ 3